<a href="https://colab.research.google.com/github/nozaniin/prog_oop/blob/lab_4_2/lab_4_2_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа 4.1
Выполнила: Гафурова Н.А

Группа: ЦИБ-251

# Реализация варианта №6 (Банковская система)
На основе кода из ЛР 4.1 созданы:

***Дочерние классы для Account:***

CheckingAccount — текущий счёт с лимитом овердрафта (overdraft_limit).

SavingsAccount — сберегательный счёт с процентной ставкой (interest_rate).


***Дочерний класс для Client:***

PremiumClient — премиум-клиент с персональным менеджером (personal_manager).


***Полиморфный метод apply_monthly_fee()***

Для CheckingAccount — ежемесячная комиссия 150 руб.

Для SavingsAccount — комиссия 50 руб.


***Класс-контейнер Portfolio***

Хранит ссылку на клиента и список счетов.


Метод add_account(account) добавляет счёт.


Метод calculate_total_fees():


Суммирует комиссии всех счетов (вызывая apply_monthly_fee()).
Если клиент является PremiumClient, общая комиссия уменьшается на 50%.

In [3]:
# ==================== Базовые классы  ====================

class Account:
    """Базовый класс банковского счёта."""
    def __init__(self, balance: float):
        if balance < 0:
            raise ValueError("Баланс не может быть отрицательным.")
        self.balance = balance

    def __str__(self):
        return f"Банковский счёт | Баланс: {self.balance:.2f} руб."

    # Полиморфный метод
    def apply_monthly_fee(self):
        """Возвращает сумму ежемесячной комиссии."""
        raise NotImplementedError("Метод должен быть переопределён в дочернем классе")


class Client:
    """Базовый класс клиента банка."""
    def __init__(self, inn: str):
        if not (inn.isdigit() and len(inn) in (10, 12)):
            raise ValueError("ИНН должен содержать 10 или 12 цифр.")
        self.inn = inn

    def __str__(self):
        return f"Клиент | ИНН: {self.inn}"


# ==================== Дочерние классы ====================

class CheckingAccount(Account):
    """Текущий счёт с овердрафтом."""
    def __init__(self, balance: float, overdraft_limit: float):
        super().__init__(balance)          # инициализация баланса через родителя
        self.overdraft_limit = overdraft_limit

    def apply_monthly_fee(self):
        """Комиссия за обслуживание текущего счёта."""
        return 150.0

    def __str__(self):
        return (f"Текущий счёт | Баланс: {self.balance:.2f} руб., "
                f"Овердрафт: {self.overdraft_limit:.2f} руб.")


class SavingsAccount(Account):
    """Сберегательный счёт с процентной ставкой."""
    def __init__(self, balance: float, interest_rate: float):
        super().__init__(balance)
        self.interest_rate = interest_rate

    def apply_monthly_fee(self) -> float:
        """Комиссия за обслуживание сберегательного счёта."""
        return 50.0

    def __str__(self):
        return (f"Сберегательный счёт | Баланс: {self.balance:.2f} руб., "
                f"Ставка: {self.interest_rate*100:.2f}%")


class PremiumClient(Client):
    """Премиум-клиент с персональным менеджером."""
    def __init__(self, inn: str, personal_manager: str):
        super().__init__(inn)
        self.personal_manager = personal_manager

    def __str__(self):
        return f"Премиум-клиент | ИНН: {self.inn}, Менеджер: {self.personal_manager}"


# ==================== Класс-контейнер ====================

class Portfolio:
    """Портфель счетов, принадлежащих клиенту."""
    def __init__(self, client: Client):
        self.client = client
        self.accounts = []          # список объектов Account

    def add_account(self, account: Account):
        """Добавить счёт в портфель."""
        self.accounts.append(account)

    def calculate_total_fees(self) -> float:
        """
        Рассчитать общую сумму комиссий за месяц.
        Если клиент премиум-класса - дать скидку 50%.
        """
        total = 0.0
        for acc in self.accounts:
            total += acc.apply_monthly_fee()   # полиморфный вызов

        # Применяем скидку для премиум-клиентов
        if isinstance(self.client, PremiumClient):
            total *= 0.5   # 50% скидка
        return total

    def display_report(self):
        """Выводит детализированный отчёт."""
        print(f"Портфель клиента: {self.client}")
        print("Счета:")
        for acc in self.accounts:
            print(f"  - {acc}")
        print(f"Общая комиссия за месяц: {self.calculate_total_fees():.2f} руб.")


# ==================== Демонстрация ====================

if __name__ == "__main__":
    print("=== Создание обычного клиента и его портфеля ===")
    client_regular = Client("1234567890")
    portfolio1 = Portfolio(client_regular)
    portfolio1.add_account(CheckingAccount(10000.0, 500.0))
    portfolio1.add_account(SavingsAccount(50000.0, 0.05))
    portfolio1.display_report()

    print("\n=== Создание премиум-клиента и его портфеля ===")
    premium = PremiumClient("987654321098", "Иван Петров")
    portfolio2 = Portfolio(premium)
    portfolio2.add_account(CheckingAccount(20000.0, 1000.0))
    portfolio2.add_account(SavingsAccount(100000.0, 0.03))
    portfolio2.add_account(CheckingAccount(5000.0, 200.0))
    portfolio2.display_report()

    # Демонстрация обработки ошибок
    print("\n=== Проверка валидации ===")
    try:
        bad_acc = CheckingAccount(-100, 50)
    except ValueError as e:
        print(f"Ошибка: {e}")

=== Создание обычного клиента и его портфеля ===
Портфель клиента: Клиент | ИНН: 1234567890
Счета:
  - Текущий счёт | Баланс: 10000.00 руб., Овердрафт: 500.00 руб.
  - Сберегательный счёт | Баланс: 50000.00 руб., Ставка: 5.00%
Общая комиссия за месяц: 200.00 руб.

=== Создание премиум-клиента и его портфеля ===
Портфель клиента: Премиум-клиент | ИНН: 987654321098, Менеджер: Иван Петров
Счета:
  - Текущий счёт | Баланс: 20000.00 руб., Овердрафт: 1000.00 руб.
  - Сберегательный счёт | Баланс: 100000.00 руб., Ставка: 3.00%
  - Текущий счёт | Баланс: 5000.00 руб., Овердрафт: 200.00 руб.
Общая комиссия за месяц: 175.00 руб.

=== Проверка валидации ===
Ошибка: Баланс не может быть отрицательным.
